# MDE calculation template - Onprem & GBQ metrics

## Imports

In [14]:
from exp_dwh_client.core.gbq.gbq_query_executor import GBQQueryExecutor
from exp_dwh_client import DWHClient

from expmetrics.models.stats.summary_statistics import SummaryStatistics
from expmetrics.statistical_methods.mde import calculate_minimum_detectable_effect

import datetime
import pandas as pd
import pytz
import numpy as np

In [15]:
gbq_query_executor = GBQQueryExecutor()

gbq_client = DWHClient(gbq_query_executor)

## Functions

based of the example in exp metrics repository

In [16]:
def get_apply_until(days_from_today=5):
    current_utc_time = datetime.datetime.now(pytz.utc)
    two_hours_ahead_zone = pytz.FixedOffset(120)  # 120 minutes = 2 hours
    current_local_time = current_utc_time.astimezone(two_hours_ahead_zone)
    apply_until = current_local_time.replace(hour=23, minute=59, second=59, microsecond=0)
    apply_until = apply_until - datetime.timedelta(days=days_from_today)
    return apply_until


def get_apply_from(apply_until, weeks_diff=1):
    days_diff = weeks_diff * 7 - 1
    apply_from = apply_until - datetime.timedelta(days=days_diff)
    apply_from = apply_from.replace(hour=0, minute=0, second=0, microsecond=0)
    return apply_from


def rename_stats_results(df, metric):
    return df.rename(
        columns={
            metric.sample_size_alias: "sample_size",
            metric.metric_average_alias: "metric_average",
            metric.standard_deviation_alias: "standard_deviation",
        }
    )


def add_mde(df, metric, number_of_variants, alpha=0.05, beta=0.2, distribution=100):
    for j, row in df.iterrows():
        assert distribution <= 100, "Distribution should be less than or equal to 100"
        assert distribution > 1, "Distribution should be a number between 1 and 100"

        min_variant_sample_size = int(row["sample_size"] / number_of_variants * distribution / 100)
        df.loc[j, "Variant sample size"] = min_variant_sample_size
        df.loc[j, "sample_size"] = int(row["sample_size"] * distribution / 100)

        control_summary = SummaryStatistics(
            sample_size=min_variant_sample_size,
            metric_average=row["metric_average"],
            standard_deviation=row["standard_deviation"],
        )
        rmde = calculate_minimum_detectable_effect(
            metric_type=metric.metric_type,
            observed_samples={
                "control_sample_size": control_summary.sample_size,
            },
            control_summary=control_summary,
            alpha=alpha,  # probability of false positive (type I error)
            beta=beta,  # probability of false negative (type II error)
        )

        df.loc[j, "Abs MDE"] = rmde
        if control_summary.metric_average == 0:
            df.loc[j, "Rel MDE (%)"] = np.nan
        else:
            df.loc[j, "Rel MDE (%)"] = rmde / float(control_summary.metric_average) * 100
    return df


async def get_mde_df(
    metric,
    number_of_variants=2,
    alpha=0.05,
    beta=0.2,
    distribution=100,
    apply_until=None,
    max_weeks_diff=2,
):
    if apply_until is None:
        apply_until = get_apply_until(metric.lag_in_days + 5)
    df_combined = pd.DataFrame()
    for weeks_diff in range(1, max_weeks_diff + 1):
        apply_from = get_apply_from(apply_until, weeks_diff=weeks_diff)
        metric_stats_query = metric.aggregate_stats_query_v2(
            apply_from=apply_from, apply_until=apply_until,
        ).get_sql(quote_char="`")

        print(metric_stats_query)

        df = await GBQQueryExecutor().execute_query(metric_stats_query)

        df["experiment duration in weeks"] = weeks_diff
        df["metric"] = metric.user_friendly_name
        df = rename_stats_results(df, metric)
        df = add_mde(
            df,
            metric,
            number_of_variants=number_of_variants,
            alpha=alpha,
            beta=beta,
            distribution=distribution,
        )
        df_combined = pd.concat([df_combined, df], ignore_index=True)

    return df_combined


## Set up

In [17]:
APPLY_UNTIL = datetime.datetime(2025, 6, 15, 0, 0, 0, tzinfo=datetime.timezone.utc)

In [18]:
from expmetrics.metrics.metric_registry import MetricRegistry
from expmetrics.datasets.segment import BareSegmentDataset


In [19]:
metric_registry = MetricRegistry()

In [20]:
from pypika import Criterion
from expmetrics.datasets.dataset_definition_params import DatasetDefinitionParams
from expmetrics.filters.filters.date_filters import time_range_filter
from pypika.queries import QueryBuilder
from expmetrics.datasets import Payouts
from expmetrics.models.marketplace_side import MarketplaceSide
from pypika import Criterion, Field, Table

class Local(Payouts):
    @property
    def dimensions(self) -> dict[str, Field]:
        return self._build_dimensions(
            [
                "user_id",
                "payout_id",
                "status_code",
                "is_failed",
                "processed_at",
                "created_at",
                "amount_eur",
                "payment_provider"
            ],
            prefix=self.prefix,
        )

    def build_filters(self, params: DatasetDefinitionParams) -> Criterion:
        filter_builder = self.init_filter_builder(params).with_time_range_filter(
            self.dimension_aliases["utc_unixtime"], filter_function=time_range_filter
        )

        # In case you need to further filter the dataset, you can add custom filters here.
        # The column name needs to have a prefix defined. It's a bit hacky, but the prefix needs to be later defined in the next cell too.
        filter_builder._custom_filters.append(self.dimensions["a__payment_provider"].isin(["adyen","vinted_pay", "mangopay"]))
        filter_builder._custom_filters.append(self.dimensions["a__amount_eur"]>500)

        filters = filter_builder.build()
        return Criterion.all(filters)
    
    def build_dataset_definition(self, params: DatasetDefinitionParams) -> QueryBuilder:

        return super().build_dataset_definition(params).distinct()


In [21]:
metric = metric_registry.get_by_alias("users_with_payout_attempt")
metric.segment_dataset = Local(prefix="a_")


In [22]:
await get_mde_df(metric=metric, apply_until=APPLY_UNTIL, max_weeks_diff=5)

SELECT AVG(`sq3`.`users_with_payout_attempt`) `avg_users_with_payout_attempt`,STDDEV(`sq3`.`users_with_payout_attempt`) `std_users_with_payout_attempt`,COUNT(`sq3`.`users_with_payout_attempt`) `cnt_users_with_payout_attempt` FROM (SELECT `sq2`.`a__user_id`,COUNT(DISTINCT `payouts_user_id`) `users_with_payout_attempt` FROM (SELECT `a__user_id`,`payouts_user_id` FROM (SELECT DISTINCT `user_id` `a__user_id` FROM `vi-dv-prod-data.marts`.`mrt_payouts` WHERE `created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600) AND `payment_provider` IN ('adyen','vinted_pay','mangopay') AND `amount_eur`>500) `sq0` LEFT JOIN (SELECT `user_id` `payouts_user_id` FROM `vi-dv-prod-data.marts`.`mrt_payouts` WHERE `created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600)) `sq1` ON `a__user_id`=`payouts_user_id`) `sq2` GROUP BY 1) `sq3`
SELECT AVG(`sq3`.`users_with_payout_attempt`) `avg_users_with_payout_attempt`,STDDEV(`sq3`.`users_with_payout_attempt`) `std

,metric_average,standard_deviation,sample_size,experiment duration in weeks,metric,Variant sample size,Abs MDE,Rel MDE (%)
0,0.999925,0.008643,13387,1,Users With Payout Attempt,6693.0,0.000419,0.041860
1,0.999963,0.006087,26986,2,Users With Payout Attempt,13493.0,0.000208,0.020764
2,0.999975,0.004981,40302,3,Users With Payout Attempt,20151.0,0.000139,0.013903
3,0.999981,0.004374,52273,4,Users With Payout Attempt,26136.0,0.000107,0.010719
4,0.999984,0.003975,63274,5,Users With Payout Attempt,31637.0,0.000089,0.008856


In [26]:
metric = metric_registry.get_by_alias("transactions_sold")
metric.segment_dataset = Local(prefix="a_")
await get_mde_df(metric=metric, apply_until=APPLY_UNTIL, max_weeks_diff=5)

SELECT AVG(`sq3`.`transactions_sold`) `avg_transactions_sold`,STDDEV(`sq3`.`transactions_sold`) `std_transactions_sold`,COUNT(`sq3`.`transactions_sold`) `cnt_transactions_sold` FROM (SELECT `sq2`.`a__user_id`,COUNT(CASE WHEN true THEN `transactions_transaction_id` END) `transactions_sold` FROM (SELECT `a__user_id`,`transactions_transaction_id`,`transactions_user_id_seller` FROM (SELECT DISTINCT `user_id` `a__user_id` FROM `vi-dv-prod-data.marts`.`mrt_payouts` WHERE `created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600) AND `payment_provider` IN ('adyen','vinted_pay','mangopay') AND `amount_eur`>500) `sq0` LEFT JOIN (SELECT `transaction_id` `transactions_transaction_id`,`user_id_seller` `transactions_user_id_seller` FROM `vi-dv-prod-data.marts`.`mrt_transactions` WHERE `transaction_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600)) `sq1` ON `a__user_id`=`transactions_user_id_seller`) `sq2` GROUP BY 1) `sq3`
SELECT AVG(`sq3`.`transa

,metric_average,standard_deviation,sample_size,experiment duration in weeks,metric,Variant sample size,Abs MDE,Rel MDE (%)
0,8.072234,17.337591,13387,1,Transactions Sold,6693.0,0.839648,10.401678
1,16.551619,32.741070,26986,2,Transactions Sold,13493.0,1.116754,6.747097
2,24.027666,44.856263,40302,3,Transactions Sold,20151.0,1.251969,5.210532
3,29.022249,53.929831,52273,4,Transactions Sold,26136.0,1.321686,4.554045
4,34.059171,62.746904,63274,5,Transactions Sold,31637.0,1.397698,4.103736


In [28]:
metric = metric_registry.get_by_alias("buyers")
metric.segment_dataset = Local(prefix="a_")
await get_mde_df(metric=metric, apply_until=APPLY_UNTIL, max_weeks_diff=5)

SELECT AVG(`sq3`.`buyers`) `avg_buyers`,STDDEV(`sq3`.`buyers`) `std_buyers`,COUNT(`sq3`.`buyers`) `cnt_buyers` FROM (SELECT `sq2`.`a__user_id`,COUNT(DISTINCT CASE WHEN true THEN `transactions_user_id_buyer` END)/NULLIF(COUNT(DISTINCT `a__user_id`),0) `buyers` FROM (SELECT `a__user_id`,`transactions_user_id_buyer` FROM (SELECT DISTINCT `user_id` `a__user_id` FROM `vi-dv-prod-data.marts`.`mrt_payouts` WHERE `created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600) AND `payment_provider` IN ('adyen','vinted_pay','mangopay') AND `amount_eur`>500) `sq0` LEFT JOIN (SELECT `user_id_buyer` `transactions_user_id_buyer` FROM `vi-dv-prod-data.marts`.`mrt_transactions` WHERE `transaction_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600)) `sq1` ON `a__user_id`=`transactions_user_id_buyer`) `sq2` GROUP BY 1) `sq3`
SELECT AVG(`sq3`.`buyers`) `avg_buyers`,STDDEV(`sq3`.`buyers`) `std_buyers`,COUNT(`sq3`.`buyers`) `cnt_buyers` FROM (SELECT `sq2`.`a__u

,metric_average,standard_deviation,sample_size,experiment duration in weeks,metric,Variant sample size,Abs MDE,Rel MDE (%)
0,0.203571,0.402668,13386,1,Buyers,6693.0,0.019145,9.404433
1,0.303465,0.459763,26985,2,Buyers,13492.0,0.015564,5.128689
2,0.360884,0.480263,40301,3,Buyers,20150.0,0.013348,3.698683
3,0.407446,0.491364,52272,4,Buyers,26136.0,0.012012,2.948228
4,0.442369,0.496671,63273,5,Buyers,31636.0,0.011048,2.497436


In [ ]:
metric = metric_registry.get_by_alias("listings_per_lister")
metric.segment_dataset = Local(prefix="a_")
await get_mde_df(metric=metric, apply_until=APPLY_UNTIL, max_weeks_diff=5)

SELECT AVG(`sq3`.`listings_per_lister`) `avg_listings_per_lister`,STDDEV(`sq3`.`listings_per_lister`) `std_listings_per_lister`,COUNT(`sq3`.`listings_per_lister`) `cnt_listings_per_lister` FROM (SELECT `sq2`.`a__user_id`,COUNT(`item_id`)/NULLIF(COUNT(DISTINCT `user_id_lister`),0) `listings_per_lister` FROM (SELECT `a__user_id`,`item_id`,`user_id_lister` FROM (SELECT DISTINCT `user_id` `a__user_id` FROM `vi-dv-prod-data.marts`.`mrt_payouts` WHERE `created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600) AND `payment_provider` IN ('adyen','vinted_pay','mangopay') AND `amount_eur`>500) `sq0` JOIN (SELECT `item_id` `item_id`,`user_id_lister` `user_id_lister` FROM `vi-dv-prod-data.marts`.`mrt_enriched_listings` WHERE `item_created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600)) `sq1` ON `a__user_id`=`user_id_lister`) `sq2` GROUP BY 1) `sq3`
SELECT AVG(`sq3`.`listings_per_lister`) `avg_listings_per_lister`,STDDEV(`sq3`.`listings_per_lis

,metric_average,standard_deviation,sample_size,experiment duration in weeks,metric,Variant sample size,Abs MDE,Rel MDE (%)
0,32.752051,131.889233,8409,1,Listings per Lister,4204.0,8.059291,24.606980
1,54.089445,233.956753,20493,2,Listings per Lister,10246.0,9.157507,16.930302
2,72.556123,335.587989,33115,3,Listings per Lister,16557.0,10.333190,14.241651
3,85.518998,409.402347,44688,4,Listings per Lister,22344.0,10.851473,12.688962
4,97.290475,466.182419,55609,5,Listings per Lister,27804.0,11.076973,11.385465


In [29]:
metric = metric_registry.get_by_alias("listers_per_au")
metric.segment_dataset = Local(prefix="a_")
await get_mde_df(metric=metric, apply_until=APPLY_UNTIL, max_weeks_diff=5)

SELECT AVG(`sq3`.`listers_per_au`) `avg_listers_per_au`,STDDEV(`sq3`.`listers_per_au`) `std_listers_per_au`,COUNT(`sq3`.`listers_per_au`) `cnt_listers_per_au` FROM (SELECT `sq2`.`a__user_id`,COUNT(DISTINCT CASE WHEN NOT `item_created_at` IS NULL THEN `a__user_id` END)/NULLIF(COUNT(DISTINCT `a__user_id`),0) `listers_per_au` FROM (SELECT `a__user_id`,`item_created_at`,`user_id_lister` FROM (SELECT DISTINCT `user_id` `a__user_id` FROM `vi-dv-prod-data.marts`.`mrt_payouts` WHERE `created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600) AND `payment_provider` IN ('adyen','vinted_pay','mangopay') AND `amount_eur`>500) `sq0` LEFT JOIN (SELECT `item_created_at` `item_created_at`,`user_id_lister` `user_id_lister` FROM `vi-dv-prod-data.marts`.`mrt_enriched_listings` WHERE `item_created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600)) `sq1` ON `a__user_id`=`user_id_lister`) `sq2` GROUP BY 1) `sq3`
SELECT AVG(`sq3`.`listers_per_au`) `avg_liste

,metric_average,standard_deviation,sample_size,experiment duration in weeks,metric,Variant sample size,Abs MDE,Rel MDE (%)
0,0.628194,0.483305,13386,1,Listers per AU,6693.0,0.023242,3.699796
1,0.759422,0.427442,26985,2,Listers per AU,13492.0,0.014425,1.899533
2,0.821692,0.382777,40301,3,Listers per AU,20150.0,0.010557,1.284799
3,0.854913,0.352192,52272,4,Listers per AU,26136.0,0.008524,0.997070
4,0.878874,0.326276,63273,5,Listers per AU,31636.0,0.007174,0.816232


In [ ]:
metric = metric_registry.get_by_alias("sellers")
metric.segment_dataset = Local(prefix="a_")
await get_mde_df(metric=metric, apply_until=APPLY_UNTIL, max_weeks_diff=5)

SELECT AVG(`sq3`.`sellers`) `avg_sellers`,STDDEV(`sq3`.`sellers`) `std_sellers`,COUNT(`sq3`.`sellers`) `cnt_sellers` FROM (SELECT `sq2`.`a__user_id`,COUNT(DISTINCT CASE WHEN true THEN `transactions_user_id_seller` END)/NULLIF(COUNT(DISTINCT `a__user_id`),0) `sellers` FROM (SELECT `a__user_id`,`transactions_user_id_seller` FROM (SELECT DISTINCT `user_id` `a__user_id` FROM `vi-dv-prod-data.marts`.`mrt_payouts` WHERE `created_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600) AND `payment_provider` IN ('adyen','vinted_pay','mangopay') AND `amount_eur`>500) `sq0` LEFT JOIN (SELECT `user_id_seller` `transactions_user_id_seller` FROM `vi-dv-prod-data.marts`.`mrt_transactions` WHERE `transaction_at` BETWEEN TIMESTAMP_SECONDS(1749427200) AND TIMESTAMP_SECONDS(1749945600)) `sq1` ON `a__user_id`=`transactions_user_id_seller`) `sq2` GROUP BY 1) `sq3`
SELECT AVG(`sq3`.`sellers`) `avg_sellers`,STDDEV(`sq3`.`sellers`) `std_sellers`,COUNT(`sq3`.`sellers`) `cnt_sellers` FROM (

,metric_average,standard_deviation,sample_size,experiment duration in weeks,metric,Variant sample size,Abs MDE,Rel MDE (%)
0,0.693112,0.461220,13386,1,Sellers,6693.0,0.022098,3.188169
1,0.830795,0.374940,26985,2,Sellers,13492.0,0.012594,1.515938
2,0.893080,0.309016,40301,3,Sellers,20150.0,0.008472,0.948580
3,0.919058,0.272749,52272,4,Sellers,26136.0,0.006559,0.713633
4,0.936640,0.243612,63273,5,Sellers,31636.0,0.005319,0.567843
